# GenAge → Knowledge Graph (KG) Builder

**Source:** [GenAge](https://genomics.senescence.info/genes/) — Model Organisms database (`genage_models.csv`)  
**Species covered:** *Saccharomyces cerevisiae*, *Caenorhabditis elegans*, *Drosophila melanogaster*, *Mus musculus*, *Homo sapiens*


## Output files
```
Human/   Human_Gene_BiologicalProcess
Yeast/   Yeast_Gene_Inhibits_BiologicalProcess.csv
Yeast/   Yeast_Gene_Promotes_BiologicalProcess.csv
Cele/    Cele_Gene_Inhibits_BiologicalProcess.csv
Cele/    Cele_Gene_Promotes_BiologicalProcess.csv
Droso/   Droso_Gene_Inhibits_BiologicalProcess.csv
Droso/   Droso_Gene_Promotes_BiologicalProcess.csv
Mouse/   Mouse_Gene_Inhibits_BiologicalProcess.csv
Mouse/   Mouse_Gene_Promotes_BiologicalProcess.csv
```



---
## 0 · Configuration — edit ONLY these two lines

In [16]:
import os
import pandas as pd
import numpy as np
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : root folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage"

# ── Derived input paths (do not edit below this line) ────────────────────────
# NCBI gene_info files — one per species

NCBI_YEAST_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Saccharomyces_cerevisiae.gene_info")
NCBI_CELE_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info")
NCBI_DROSO_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Drosophila_melanogaster.gene_info")
NCBI_MOUSE_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Mus_musculus.gene_info")
# GenAge model organisms dataset
GENAGE_PATH      = os.path.join(BASE_PATH, "genage/genage_models.csv")
# GenDR dietary restriction manipulations dataset
GENDR_PATH       = os.path.join(BASE_PATH, "gendr/cleaned_gendr_manipulations.csv")

# ── Derived output directories ───────────────────────────────────────────────
OUT_YEAST = os.path.join(OUT_PATH, "Yeast/")
OUT_CELE  = os.path.join(OUT_PATH, "Cele/")
OUT_DROSO = os.path.join(OUT_PATH, "Droso/")
OUT_MOUSE = os.path.join(OUT_PATH, "Mouse/")
OUT_Human = os.path.join(OUT_PATH, "Human/")

for d in [OUT_YEAST, OUT_CELE, OUT_DROSO, OUT_MOUSE]:
    os.makedirs(d, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output root : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output root : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage


---
## 1 · Load NCBI gene reference dictionaries (per species)

Each species needs different lookups because GenAge uses different identifier types:

| Species | GenAge identifier | Normalisation needed |
|---|---|---|
 Human | Gene Symbol or LocusTag | Symbol  → description |
| Yeast | Gene Symbol or LocusTag | Symbol → LocusTag → description |
| C. elegans | LocusTag with `CELE_` prefix | Strip prefix → Symbol → description |
| Drosophila | LocusTag with `Dmel_` prefix | Strip prefix → Symbol → description |
| Mouse | Gene Symbol (capitalised) | Capitalise first letter → description |

In [5]:
# ── 1 Human NCBI Gene Info ───────────────────────────────────────────────────
# Official NCBI gene_info file for Homo sapiens
# Columns of interest: GeneID, Symbol, Synonyms, description
NCBI_ALL_GENE = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Homo_sapiens.gene_info',
    sep='\t'
)

# GeneID → gene Symbol       (used to convert numeric IDs back to readable symbols)
NCBI_ALL_GENEIDname_dict  = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))

# Gene Symbol → gene description  (direct symbol-based lookup, no ID needed)
NCBI_ALL_Symb_Desc_dict   = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))


In [2]:
# ── 1a. Yeast (Saccharomyces cerevisiae) ──────────────────────────────────────
NCBI_Yeast_gene = pd.read_csv(NCBI_YEAST_PATH, sep='\t')

# Symbol → LocusTag  (e.g. 'SIR2' → 'YDL042C')
NCBI_ALL_GENEname_dict         = dict(zip(NCBI_Yeast_gene['Symbol'],   NCBI_Yeast_gene['LocusTag']))
# Symbol → Full name description
NCBI_GENEname_Description_dict = dict(zip(NCBI_Yeast_gene['Symbol'],   NCBI_Yeast_gene['Full_name_from_nomenclature_authority']))
# LocusTag → Full name description  (fallback when Symbol lookup fails)
NCBI_LocusTag_Description_dict = dict(zip(NCBI_Yeast_gene['LocusTag'], NCBI_Yeast_gene['Full_name_from_nomenclature_authority']))

print(f"Yeast: {len(NCBI_GENEname_Description_dict):,} symbol→description, {len(NCBI_LocusTag_Description_dict):,} locustag→description")
NCBI_Yeast_gene.head(3)

Yeast: 6,513 symbol→description, 6,513 locustag→description


,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,4932,2828223,NEWENTRY,-,-,-,-,-,Record to support submission of GeneRIFs for a...,other,-,-,-,-,20241011,-
1,4932,24573110,atp6,ACI60_gp06,-,-,MT,-,Atp6,protein-coding,-,-,-,-,20230818,-
2,4932,24573111,ACI60_gt03,ACI60_gt03,-,-,MT,-,tRNA-Glu,tRNA,-,-,-,-,20150619,-


In [4]:
# ── 1b. C. elegans ───────────────────────────────────────────────────────────
NCBI_Cele_gene = pd.read_csv(NCBI_CELE_PATH, sep='\t')

# Strip the 'CELE_' prefix from LocusTags to match GenAge identifiers
# e.g. 'CELE_R07B1.9' → 'R07B1.9'
NCBI_Cele_gene['LocusTag'] = NCBI_Cele_gene['LocusTag'].str.replace('CELE_', '', regex=False)

# LocusTag → canonical Gene Symbol  (e.g. 'R07B1.9' → 'aak-2')
CELE_LocusTag_2_Symbol_dict      = dict(zip(NCBI_Cele_gene['LocusTag'], NCBI_Cele_gene['Symbol']))
# Symbol → description
NCBI_Cele_Symbol_Description_dict = dict(zip(NCBI_Cele_gene['Symbol'],  NCBI_Cele_gene['description']))

print(f"C. elegans: {len(CELE_LocusTag_2_Symbol_dict):,} locustag→symbol, {len(NCBI_Cele_Symbol_Description_dict):,} symbol→description")

C. elegans: 46,927 locustag→symbol, 46,927 symbol→description


In [5]:
# ── 1c. Drosophila melanogaster ───────────────────────────────────────────────
NCBI_Droso_gene = pd.read_csv(NCBI_DROSO_PATH, sep='\t')

# Strip the 'Dmel_' prefix from LocusTags to match GenAge identifiers
# e.g. 'Dmel_CG5362' → 'CG5362'
NCBI_Droso_gene['LocusTag'] = NCBI_Droso_gene['LocusTag'].str.replace('Dmel_', '', regex=False)

# LocusTag → canonical Gene Symbol  (e.g. 'CG5362' → 'Atg7')
Droso_LocusTag_2_Symbol_dict        = dict(zip(NCBI_Droso_gene['LocusTag'], NCBI_Droso_gene['Symbol']))
# Symbol → description
NCBI_Droso_Symbol_Description_dict  = dict(zip(NCBI_Droso_gene['Symbol'],   NCBI_Droso_gene['description']))

print(f"Drosophila: {len(Droso_LocusTag_2_Symbol_dict):,} locustag→symbol, {len(NCBI_Droso_Symbol_Description_dict):,} symbol→description")

Drosophila: 17,873 locustag→symbol, 25,079 symbol→description


In [6]:
# ── 1d. Mouse (Mus musculus) ──────────────────────────────────────────────────
# Mouse LocusTags do NOT have a 'Dmel_' prefix — this was a copy-paste error
# from the Drosophila cell. Mouse LocusTags are loaded as-is.
NCBI_Mouse_gene = pd.read_csv(NCBI_MOUSE_PATH, sep='\t')

# Symbol → description
NCBI_Mouse_Symbol_Description_dict = dict(zip(NCBI_Mouse_gene['Symbol'], NCBI_Mouse_gene['description']))

print(f"Mouse: {len(NCBI_Mouse_Symbol_Description_dict):,} symbol→description")

Mouse: 112,035 symbol→description


---
## 2 · Load and process GenAge model organisms data

**Source:** `genage_models.csv`  
**Key decision:** `longevity influence` column is used for relation direction.  
Rows with `avg lifespan change (max obsv)` = NaN are **kept** — the longevity influence  
label alone is sufficient for a causal KG edge (see inline comment).

**Longevity → KG Relation mapping:**
- `Pro-Longevity`  → gene extends lifespan → **inhibits** aging process → `Gene_Inhibits_BiologicalProcess`
- `Anti-Longevity` → gene shortens lifespan → **promotes** aging process → `Gene_Promotes_BiologicalProcess`

In [7]:
# ── 2a. Load GenAge model organism genes ─────────────────────────────────────
model_gene = pd.read_csv(GENAGE_PATH)
print(f"Raw: {len(model_gene):,} rows")
print("Organisms:", set(model_gene['organism']))
print("Longevity values:", set(model_gene['longevity influence']))
model_gene

Raw: 2,205 rows
Organisms: {'Saccharomyces cerevisiae', 'Danio rerio', 'Caenorhabditis briggsae', 'Mus musculus', 'Caenorhabditis elegans', 'Schizosaccharomyces pombe', 'Mesocricetus auratus', 'Drosophila melanogaster', 'Podospora anserina'}
Longevity values: {'Anti-Longevity', 'Necessary for fitness', 'Unclear', 'Pro-Longevity', 'Unannotated'}


,GenAge ID,symbol,name,organism,entrez gene id,avg lifespan change (max obsv),lifespan effect,longevity influence
0,1,aak-2,AMP-Activated Kinase,Caenorhabditis elegans,181727.0,NaN,Increase and Decrease,Pro-Longevity
1,2,aat-8,Amino Acid Transporter,Caenorhabditis elegans,185079.0,30.0,Increase,Anti-Longevity
2,3,abu-11,Activated in Blocked Unfolded protein response,Caenorhabditis elegans,173404.0,28.0,Increase,Pro-Longevity
3,4,acl-11,ACyLtransferase-like,Caenorhabditis elegans,185044.0,NaN,Increase,Anti-Longevity
4,5,aco-2,ACOnitase,Caenorhabditis elegans,176121.0,20.5,Increase,Anti-Longevity
...,...,...,...,...,...,...,...,...
2200,2267,SPCC794.03,SPCC794.03,Schizosaccharomyces pombe,2539472.0,NaN,Increase,Anti-Longevity
2201,2268,SPBC8E4.02c,SPBC8E4.02c,Schizosaccharomyces pombe,2541246.0,NaN,Increase,Anti-Longevity
2202,2269,SPBC18H10.18c,SPBC18H10.18c,Schizosaccharomyces pombe,2540834.0,NaN,Increase,Anti-Longevity
2203,2270,SPCC594.02c,SPCC594.02c,Schizosaccharomyces pombe,2538998.0,NaN,Increase,Anti-Longevity


In [10]:
# ── 2b. Filter to 5 target species ───────────────────────────────────────────
Sp_list = [
    'Caenorhabditis elegans', 'Drosophila melanogaster',
    'Saccharomyces cerevisiae', 'Homo sapiens', 'Mus musculus'
]
model_gene = model_gene[model_gene['organism'].isin(Sp_list)]
print(f"After species filter: {len(model_gene):,} rows")

After species filter: 2,138 rows


In [11]:
# ── 2c. Rename columns and keep only directional rows ─────────────────────────
model_gene = model_gene.rename(columns={
    'symbol':              'Head',
    'longevity influence': 'Tail',
    'organism':            'Species'
})

model_gene['Head_type']       = 'Gene'
model_gene['Tail_type']       = 'Phenotype'   # updated to BiologicalProcess below
model_gene['Relation']        = 'Gene_Phenotype'
model_gene['Relation_Source'] = 'GeneAge'

# Keep only rows with a clear directional longevity label.
# 'Unclear' and other labels are excluded — no directional KG edge can be assigned.
# NOTE: rows where 'avg lifespan change (max obsv)' is NaN are intentionally kept;
# the 'longevity influence' label is the authoritative causal annotation.
Tail_list = ['Pro-Longevity', 'Anti-Longevity']
model_gene = model_gene[model_gene['Tail'].isin(Tail_list)]
print(f"After directional filter: {len(model_gene):,} rows")
print("Tail distribution:", model_gene['Tail'].value_counts().to_dict())

After directional filter: 1,589 rows
Tail distribution: {'Anti-Longevity': 1045, 'Pro-Longevity': 544}


In [12]:
# ── 2d. Map longevity labels → canonical KG Relation; pin Tail to GO:0007568 ─
# Pro-Longevity  : gene extends lifespan → it INHIBITS the aging biological process
# Anti-Longevity : gene shortens lifespan → it PROMOTES the aging biological process
model_gene['Tail'] = model_gene['Tail'].replace({
    'Pro-Longevity':  'Gene_Inhibits_BiologicalProcess',
    'Anti-Longevity': 'Gene_Promotes_BiologicalProcess'
})

# Swap Relation ↔ Tail: Relation holds edge type, Tail holds GO term
model_gene[['Relation', 'Tail']] = model_gene[['Tail', 'Relation']]

# All edges target the aging biological process GO term
model_gene['Tail']             = 'GO:0007568'
model_gene['Tail_detail_name'] = 'obsolete aging'
model_gene['Tail_type']        = 'BiologicalProcess'
model_gene['Data_type']        = 'Causative'
model_gene['Head_detail_name'] = np.nan   # filled per species below

# Select and order final columns
desired_order = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Tail_type',
    'Head_detail_name', 'Tail_detail_name',
    'Relation_Source', 'Species', 'Data_type'
]
model_gene = model_gene[desired_order]

print("Relation distribution:")
print(model_gene['Relation'].value_counts())
print("\nSpecies distribution:")
print(model_gene['Species'].value_counts())

Relation distribution:
Relation
Gene_Promotes_BiologicalProcess    1045
Gene_Inhibits_BiologicalProcess     544
Name: count, dtype: int64

Species distribution:
Species
Caenorhabditis elegans      861
Saccharomyces cerevisiae    400
Drosophila melanogaster     198
Mus musculus                130
Name: count, dtype: int64


---
## 3 · Per-species processing and export

Each species section:
1. Filters `model_gene` to that species
2. Normalises gene identifiers (LocusTag → Symbol where needed)
3. Validates against NCBI by mapping Symbol → description (drops unmatched rows)
4. Splits by relation direction and exports two files

### 3a · Yeast (*Saccharomyces cerevisiae*)

In [13]:
Yeast_Gene_BIoP = model_gene[model_gene['Species'] == 'Saccharomyces cerevisiae'].copy()
print(f"Yeast raw: {len(Yeast_Gene_BIoP):,} rows")

# Yeast GenAge entries may be Gene Symbols OR LocusTags.
# Try Symbol lookup first; fall back to LocusTag lookup.
Yeast_Gene_BIoP['Head_detail_name'] = (
    Yeast_Gene_BIoP['Head'].map(NCBI_GENEname_Description_dict)
    .fillna(Yeast_Gene_BIoP['Head'].map(NCBI_LocusTag_Description_dict))
)

# Map Symbol → LocusTag as an alternate identifier (kept for traceability)
Yeast_Gene_BIoP['Head_alt_name'] = Yeast_Gene_BIoP['Head'].map(NCBI_ALL_GENEname_dict).fillna(Yeast_Gene_BIoP['Head'])

# Drop rows where neither lookup resolved a description
before = len(Yeast_Gene_BIoP)
Yeast_Gene_BIoP = Yeast_Gene_BIoP[~Yeast_Gene_BIoP['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Yeast_Gene_BIoP):,} kept / {before - len(Yeast_Gene_BIoP):,} dropped")
print(Yeast_Gene_BIoP['Relation'].value_counts())

Yeast raw: 400 rows
NCBI validated: 397 kept / 3 dropped
Relation
Gene_Promotes_BiologicalProcess    347
Gene_Inhibits_BiologicalProcess     50
Name: count, dtype: int64


In [17]:
Gene_Promotes_BiologicalProcess = Yeast_Gene_BIoP[Yeast_Gene_BIoP['Relation'] == 'Gene_Promotes_BiologicalProcess']
Gene_Inhibits_BiologicalProcess = Yeast_Gene_BIoP[Yeast_Gene_BIoP['Relation'] == 'Gene_Inhibits_BiologicalProcess']

Gene_Promotes_BiologicalProcess.to_csv(os.path.join(OUT_YEAST, 'Yeast_Gene_Promotes_BiologicalProcess.csv'), index=False)
Gene_Inhibits_BiologicalProcess.to_csv(os.path.join(OUT_YEAST, 'Yeast_Gene_Inhibits_BiologicalProcess.csv'), index=False)
print(f"Saved → {OUT_YEAST}  ({len(Gene_Promotes_BiologicalProcess):,} Promotes, {len(Gene_Inhibits_BiologicalProcess):,} Inhibits)")

Saved → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage/Yeast/  (347 Promotes, 50 Inhibits)


### 3b · C. elegans (*Caenorhabditis elegans*)

In [18]:
Cele_Gene_BIoP = model_gene[model_gene['Species'] == 'Caenorhabditis elegans'].copy()
print(f"C. elegans raw: {len(Cele_Gene_BIoP):,} rows")

# GenAge C. elegans entries use LocusTags with a 'CELE_' prefix (e.g. 'CELE_R07B1.9').
# Step 1: strip the prefix
Cele_Gene_BIoP['Head'] = Cele_Gene_BIoP['Head'].str.replace('CELE_', '', regex=False)

# Step 2: map LocusTag → canonical NCBI Symbol (e.g. 'R07B1.9' → 'aak-2');
# keep the original LocusTag as Head_alt_name for traceability.

# Here we use a clean separate column for the alternate name.
Cele_Gene_BIoP['Head_alt_name'] = Cele_Gene_BIoP['Head']   # save original LocusTag
Cele_Gene_BIoP['Head'] = (
    Cele_Gene_BIoP['Head'].map(CELE_LocusTag_2_Symbol_dict)
    .fillna(Cele_Gene_BIoP['Head'])   # keep LocusTag if no symbol found
)

# Step 3: validate Symbol against NCBI description dict
Cele_Gene_BIoP['Head_detail_name'] = Cele_Gene_BIoP['Head'].map(NCBI_Cele_Symbol_Description_dict)
before = len(Cele_Gene_BIoP)
Cele_Gene_BIoP = Cele_Gene_BIoP[~Cele_Gene_BIoP['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Cele_Gene_BIoP):,} kept / {before - len(Cele_Gene_BIoP):,} dropped")
print(Cele_Gene_BIoP['Relation'].value_counts())

C. elegans raw: 861 rows
NCBI validated: 835 kept / 26 dropped
Relation
Gene_Promotes_BiologicalProcess    557
Gene_Inhibits_BiologicalProcess    278
Name: count, dtype: int64


In [19]:
Gene_Promotes_BiologicalProcess = Cele_Gene_BIoP[Cele_Gene_BIoP['Relation'] == 'Gene_Promotes_BiologicalProcess']
Gene_Inhibits_BiologicalProcess = Cele_Gene_BIoP[Cele_Gene_BIoP['Relation'] == 'Gene_Inhibits_BiologicalProcess']

Gene_Promotes_BiologicalProcess.to_csv(os.path.join(OUT_CELE, 'Cele_Gene_Promotes_BiologicalProcess.csv'), index=False)
Gene_Inhibits_BiologicalProcess.to_csv(os.path.join(OUT_CELE, 'Cele_Gene_Inhibits_BiologicalProcess.csv'), index=False)
print(f"Saved → {OUT_CELE}  ({len(Gene_Promotes_BiologicalProcess):,} Promotes, {len(Gene_Inhibits_BiologicalProcess):,} Inhibits)")

Saved → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage/Cele/  (557 Promotes, 278 Inhibits)


### 3c · Drosophila (*Drosophila melanogaster*)

In [20]:
Droso_Gene_BIoP = model_gene[model_gene['Species'] == 'Drosophila melanogaster'].copy()
print(f"Drosophila raw: {len(Droso_Gene_BIoP):,} rows")

# GenAge Drosophila entries use LocusTags (e.g. 'CG5362' after the 'Dmel_' prefix is
# already stripped during NCBI dict construction above).
# Map LocusTag → canonical NCBI Symbol (e.g. 'CG5362' → 'Atg7').

Droso_Gene_BIoP['Head_alt_name'] = Droso_Gene_BIoP['Head']   # save original LocusTag
Droso_Gene_BIoP['Head'] = (
    Droso_Gene_BIoP['Head'].map(Droso_LocusTag_2_Symbol_dict)
    .fillna(Droso_Gene_BIoP['Head'])
)

# Validate Symbol against NCBI description dict
Droso_Gene_BIoP['Head_detail_name'] = Droso_Gene_BIoP['Head'].map(NCBI_Droso_Symbol_Description_dict)
before = len(Droso_Gene_BIoP)
Droso_Gene_BIoP = Droso_Gene_BIoP[~Droso_Gene_BIoP['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Droso_Gene_BIoP):,} kept / {before - len(Droso_Gene_BIoP):,} dropped")
print(Droso_Gene_BIoP['Relation'].value_counts())

Drosophila raw: 198 rows
NCBI validated: 168 kept / 30 dropped
Relation
Gene_Inhibits_BiologicalProcess    107
Gene_Promotes_BiologicalProcess     61
Name: count, dtype: int64


In [21]:
Gene_Promotes_BiologicalProcess = Droso_Gene_BIoP[Droso_Gene_BIoP['Relation'] == 'Gene_Promotes_BiologicalProcess']
Gene_Inhibits_BiologicalProcess = Droso_Gene_BIoP[Droso_Gene_BIoP['Relation'] == 'Gene_Inhibits_BiologicalProcess']

Gene_Promotes_BiologicalProcess.to_csv(os.path.join(OUT_DROSO, 'Droso_Gene_Promotes_BiologicalProcess.csv'), index=False)
Gene_Inhibits_BiologicalProcess.to_csv(os.path.join(OUT_DROSO, 'Droso_Gene_Inhibits_BiologicalProcess.csv'), index=False)
print(f"Saved → {OUT_DROSO}  ({len(Gene_Promotes_BiologicalProcess):,} Promotes, {len(Gene_Inhibits_BiologicalProcess):,} Inhibits)")

Saved → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage/Droso/  (61 Promotes, 107 Inhibits)


### 3d · Mouse (*Mus musculus*)

In [22]:
Mouse_Gene_BIoP = model_gene[model_gene['Species'] == 'Mus musculus'].copy()
print(f"Mouse raw: {len(Mouse_Gene_BIoP):,} rows")

# GenAge Mouse symbols are sometimes stored in all-uppercase (e.g. 'SIRT1').
# NCBI Mouse symbols use title-case (e.g. 'Sirt1').
# This function capitalises only the first letter without lowercasing the rest,
# so symbols already in correct case (e.g. 'Klotho') are left unchanged.
def format_mouse_symbol(value):
    """Capitalise first letter of a gene symbol if it is not already title-cased."""
    if pd.isna(value):
        return value
    # If it already starts with uppercase followed by lowercase, leave as-is
    return value if pd.Series([value]).str.match(r'^[A-Z][a-z]').bool() else value.capitalize()

Mouse_Gene_BIoP['Head'] = Mouse_Gene_BIoP['Head'].apply(format_mouse_symbol)

# Validate against Mouse NCBI description dict
Mouse_Gene_BIoP['Head_detail_name'] = Mouse_Gene_BIoP['Head'].map(NCBI_Mouse_Symbol_Description_dict)
before = len(Mouse_Gene_BIoP)
Mouse_Gene_BIoP = Mouse_Gene_BIoP[~Mouse_Gene_BIoP['Head_detail_name'].isna()]
print(f"NCBI validated: {len(Mouse_Gene_BIoP):,} kept / {before - len(Mouse_Gene_BIoP):,} dropped")
print(Mouse_Gene_BIoP['Relation'].value_counts())

Mouse raw: 130 rows
NCBI validated: 128 kept / 2 dropped
Relation
Gene_Inhibits_BiologicalProcess    83
Gene_Promotes_BiologicalProcess    45
Name: count, dtype: int64


/tmp/ipykernel_1432734/3038244567.py:13: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  return value if pd.Series([value]).str.match(r'^[A-Z][a-z]').bool() else value.capitalize()


In [23]:
Gene_Promotes_BiologicalProcess = Mouse_Gene_BIoP[Mouse_Gene_BIoP['Relation'] == 'Gene_Promotes_BiologicalProcess']
Gene_Inhibits_BiologicalProcess = Mouse_Gene_BIoP[Mouse_Gene_BIoP['Relation'] == 'Gene_Inhibits_BiologicalProcess']

Gene_Promotes_BiologicalProcess.to_csv(os.path.join(OUT_MOUSE, 'Mouse_Gene_Promotes_BiologicalProcess.csv'), index=False)
Gene_Inhibits_BiologicalProcess.to_csv(os.path.join(OUT_MOUSE, 'Mouse_Gene_Inhibits_BiologicalProcess.csv'), index=False)
print(f"Saved → {OUT_MOUSE}  ({len(Gene_Promotes_BiologicalProcess):,} Promotes, {len(Gene_Inhibits_BiologicalProcess):,} Inhibits)")

Saved → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage/Mouse/  (45 Promotes, 83 Inhibits)


# Human File

In [17]:
import pandas as pd
genage_human = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/genage/genage_human.csv')

genage_human['head']             = genage_human['symbol']
genage_human['relation']    = 'Gene_BiologicalProcess'
genage_human['tail']             = 'GO:0007568'
genage_human['head_type']        = 'Gene'
genage_human['tail_type']        = 'BiologicalProcess'
genage_human['kg_source']        = 'GenAge'
genage_human['relation_type']    = 'Causative'
genage_human['head_id_is']       = 'NCBI_ID'
genage_human['tail_id_is']       = 'Quick_GO'
genage_human['tail_detail_name'] = 'aging'
genage_human['species']          = 'Homo sapiens'

# Map head_detail_name
genage_human['head_detail_name'] = genage_human['symbol'].map(NCBI_ALL_Symb_Desc_dict)

genage_human = genage_human[~genage_human['head_detail_name'].isna()]

# Drop unmapped
genage_human = genage_human[~genage_human['head_detail_name'].isna()]
# Keep only KG columns
cols = ['head', 'relation', 'tail', 'head_type', 'tail_type','kg_source', 'kg_type', 'head_id_is', 'tail_id_is','relation_type','head_detail_name', 'tail_detail_name', 'species']

cols_present = [c for c in cols if c in genage_human.columns]
genage_human = genage_human[cols_present]

print(genage_human.shape)
genage_human.to_csv(os.path.join(OUT_Human, 'Human_Gene_BiologicalProcess.csv'), index=False)
genage_human

(302, 12)


,head,relation,tail,head_type,tail_type,kg_source,head_id_is,tail_id_is,relation_type,head_detail_name,tail_detail_name,species
0,GHR,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,growth hormone receptor,aging,Homo sapiens
1,GHRH,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,growth hormone releasing hormone,aging,Homo sapiens
2,SHC1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,SHC adaptor protein 1,aging,Homo sapiens
3,POU1F1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,POU class 1 homeobox 1,aging,Homo sapiens
4,PROP1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,PROP paired-like homeobox 1,aging,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...
302,TRAP1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,TNF receptor associated protein 1,aging,Homo sapiens
303,TRPV1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,transient receptor potential cation channel su...,aging,Homo sapiens
304,NFE2L1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,NFE2 like bZIP transcription factor 1,aging,Homo sapiens
305,IFNB1,Gene_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,GenAge,NCBI_ID,Quick_GO,Causative,interferon beta 1,aging,Homo sapiens


---
## 5 · Summary — all output files written

In [21]:
# Walk OUT_PATH and list every GenAge CSV produced with row count and file size.

# from a specific working directory. Replaced with absolute OUT_PATH.
print(f"Output files under: {OUT_PATH}\n")
total = 0
for root, dirs, files in os.walk(OUT_PATH):
    dirs.sort()
    for fname in sorted(files):
        if fname.endswith('.csv') and ('Gene' in fname or 'Gene_Inhibits' in fname):
            full = os.path.join(root, fname)
            rows = sum(1 for _ in open(full)) - 1
            size = os.path.getsize(full)
            rel  = os.path.relpath(full, OUT_PATH)
            total += 1
            print(f"  {rel:<65s}  {rows:>5,} rows  {size:>8,} bytes")
print(f"\nTotal: {total} files")

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage

  Cele/Cele_Gene_Inhibits_BiologicalProcess.csv                        278 rows    48,746 bytes
  Cele/Cele_Gene_Promotes_BiologicalProcess.csv                        557 rows    98,191 bytes
  Droso/Droso_Gene_Inhibits_BiologicalProcess.csv                      107 rows    16,633 bytes
  Droso/Droso_Gene_Promotes_BiologicalProcess.csv                       61 rows     9,423 bytes
  Human/Human_Gene_BiologicalProcess.csv                               302 rows    44,472 bytes
  Mouse/Mouse_Gene_Inhibits_BiologicalProcess.csv                       83 rows    12,784 bytes
  Mouse/Mouse_Gene_Promotes_BiologicalProcess.csv                       45 rows     7,062 bytes
  Yeast/Yeast_Gene_Inhibits_BiologicalProcess.csv                       50 rows     8,369 bytes
  Yeast/Yeast_Gene_Promotes_BiologicalProcess.csv                      347 rows    57,988 bytes

Total: 9 files


In [19]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/genage'